In [1]:
# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import pandas as pd

from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

import warnings
warnings.filterwarnings("ignore")

In [2]:
# =========================================================
# 2. LOAD PROCESSED DATA
# =========================================================

X_train = pd.read_csv(
    "../results/X_train.csv"
)

X_test = pd.read_csv(
    "../results/X_test.csv"
)

y_train = pd.read_csv(
    "../results/y_train.csv"
)["target"]

y_test = pd.read_csv(
    "../results/y_test.csv"
)["target"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4000, 133)
X_test: (1000, 133)
y_train: (4000,)
y_test: (1000,)


In [3]:
# =========================================================
# 3. XGBOOST
# =========================================================

xgb = XGBClassifier(
    random_state=42,
    eval_metric="mlogloss",
    objective="multi:softprob",
    num_class=3
)

param_grid_xgb = {
    "n_estimators": [
        100,
        200
    ],
    "max_depth": [
        3,
        6
    ],
    "learning_rate": [
        0.01,
        0.1
    ]
}

xgb_grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid_xgb,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

# Balanced sample weights
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

print("Training XGBoost...")

xgb_grid.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

best_xgb = xgb_grid.best_estimator_

y_pred_xgb = best_xgb.predict(
    X_test
)

print("\nBest Parameters:")
print(
    xgb_grid.best_params_
)

Training XGBoost...

Best Parameters:
{'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 200}


In [4]:
# =========================================================
# 4. XGBOOST EVALUATION
# =========================================================

class_names = [
    "Fair",
    "Good",
    "Poor"
]

print(
    "\nXGBoost Report:"
)

print(
    classification_report(
        y_test,
        y_pred_xgb,
        target_names=class_names
    )
)


XGBoost Report:
              precision    recall  f1-score   support

        Fair       0.95      0.78      0.86       722
        Good       0.76      0.91      0.83       255
        Poor       0.17      0.78      0.28        23

    accuracy                           0.81      1000
   macro avg       0.63      0.82      0.65      1000
weighted avg       0.89      0.81      0.84      1000

